# 🛡️ BlockAds Filter — Dead Domain Pruner on Google Colab

Notebook này chạy trực tiếp trên máy ảo **Google Colab**, tận dụng băng thông Google Cloud để quét và loại bỏ các tên miền không còn hoạt động (NXDOMAIN) khỏi danh sách bộ lọc quảng cáo.

### Bước 1: Chuẩn bị mã nguồn từ GitHub

In [ ]:
# @title 1. Cập nhật mã nguồn mới nhất
import os
if not os.path.exists("/content/blockads_filter_bin"):
    !git clone https://github.com/pass-with-high-score/blockads_filter_bin.git /content/blockads_filter_bin
%cd /content/blockads_filter_bin
!git fetch origin main
!git reset --hard origin/main


### Bước 2: Quét & lọc một bộ lọc (URL tùy ý)
Bạn có thể nhập bất kỳ URL danh sách chặn nào vào ô bên dưới.

In [ ]:
# @title 2. Quét & Lọc theo URL
filter_url = "https://www.zoso.ro/pages/rolist.txt" # @param {type:"string"}
concurrency = 600 # @param {type:"integer"}
timeout_sec = 2.0 # @param {type:"number"}
output_file = "cleaned_filter.txt" # @param {type:"string"}

!python3 scripts/colab_clean_filter.py \
    --url "$filter_url" \
    --concurrency $concurrency \
    --timeout $timeout_sec \
    --output "$output_file"

### Bước 3: Cài đặt Bun Runtime (cho Database mode)

In [ ]:
# @title 3. Cài đặt Bun & dependencies
!curl -fsSL https://bun.sh/install | bash
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.bun/bin:{os.environ['PATH']}"
!bun install

### Bước 4: Quét toàn bộ bộ lọc trong Database
> **Mẹo:** Bạn có thể lưu `DATABASE_URL` vào mục **Secrets (biểu tượng chiếc chìa khóa ở thanh bên trái)** trong Colab với tên `DATABASE_URL`.

In [ ]:
# @title 4. Chạy quét từ Database (Turbo 8x - 10x)
import getpass, os
try:
    from google.colab import userdata
    db_url = userdata.get('DATABASE_URL')
except Exception:
    db_url = os.environ.get('DATABASE_URL', '')

if not db_url:
    db_url = getpass.getpass("Nhập DATABASE_URL: ")

os.environ['DATABASE_URL'] = db_url

# Số lượng bộ lọc cần quét (chọn --limit 10 hoặc --all để quét tất cả)
limit = 10 # @param {type:"integer"}
# Số luồng chạy song song (mặc định 8 workers tăng tốc 8x - 10x)
concurrency = 8 # @param {type:"integer"}
!bun run scripts/clean-filters-db.ts --limit $limit --concurrency $concurrency


### Bước 5: Build & Upload lên Cloudflare R2 (Tuần tự: Xong cái nào Upload ngay cái đó)
> Bước này sẽ đọc từng filter từ DB, kiểm tra , nén file nhị phân  / , và **upload ngay lập tức lên Cloudflare R2** sau mỗi filter (không cần đợi hết cả 10 cái hay toàn bộ).
> Cần các Secrets trên Colab: , , , .

In [29]:
# @title 5. Build và Upload lên R2 (Chạy song song 8x - 10x Turbo Workers)
from google.colab import userdata
import os

env_keys = ["R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME", "R2_PUBLIC_URL", "DATABASE_URL"]
env_lines = []

for key in env_keys:
    val = ""
    try:
        val = userdata.get(key)
    except Exception:
        pass
    if not val:
        val = os.environ.get(key, "")
    if val:
        os.environ[key] = val
        env_lines.append(f"{key}={val}")

# Ghi file .env để Bun tự động nạp
with open(".env", "w", encoding="utf-8") as f:
    for line in env_lines:
        f.write(line + chr(10))

print(f"✓ Đã nạp {len(env_lines)} biến môi trường vào .env")

# Số lượng bộ lọc cần build (nhập 0 để build tất cả 1,864 filters)
rebuild_limit = 0 # @param {type:"integer"}
# Số luồng build & upload song song đồng thời (8 - 10 workers)
concurrency = 8 # @param {type:"integer"}

limit_arg = f"--limit {rebuild_limit}" if rebuild_limit > 0 else ""
!bun run scripts/rebuild-all.ts --concurrency $concurrency $limit_arg


✓ Đã nạp 6 biến môi trường vào .env
[2026-10-10T11:58:30.455Z] Starting rebuild of filters...
Found 1864 filters in database.

▶ [1/1864] Rebuilding '1275_dga_bamital_8776e5ad' (https://1275.ru/DGA/bamital.txt)...
[1275_dga_bamital_8776e5ad] ▶ Starting compilation: https://1275.ru/DGA/bamital.txt
[1275_dga_bamital_8776e5ad] ✓ Parsed 29118 domains, 0 CSS rules, 0 scriptlets in 0.86s
[1275_dga_bamital_8776e5ad] ✓ Trie built: 29122 nodes (1252134 bytes)
[1275_dga_bamital_8776e5ad] ✓ Bloom Filter built: 418648 bits, 10 hashes (52355 bytes)
[1275_dga_bamital_8776e5ad] ✅ Compilation complete: 29118 rules, 1305029 bytes zip package
✓ Successfully rebuilt '1275_dga_bamital_8776e5ad'

▶ [2/1864] Rebuilding '1275_dga_corebot_e8c1020f' (https://1275.ru/DGA/corebot.txt)...
[1275_dga_corebot_e8c1020f] ▶ Starting compilation: https://1275.ru/DGA/corebot.txt
[1275_dga_corebot_e8c1020f] ✓ Parsed 2516 domains, 0 CSS rules, 0 scriptlets in 0.28s
[1275_dga_corebot_e8c1020f] ✓ Trie built: 2519 nodes (7493